# 10 — Final Model Fit & Artifact Freeze (Target Cap 250)

This notebook turns the selected configuration into the project's active model artifact. It **does not run another hyperparameter search**. Based on Notebook 09, cap 250 had the lowest macro-subset MAE on all three repeated engine-level splits and the lowest mean macro-subset MAE/RMSE. Cap 350 had the lower mean NASA-style score, so the configuration below makes an explicit **MAE-first** choice while recording that trade-off in the artifact metadata.

**Fixed model configuration**
- Target cap: **250 cycles**
- Objective: `reg:pseudohubererror`, `huber_slope=10`
- Sample weighting: `row_uniform`
- Rolling window: 10 cycles
- Feature mode: regime-aware (KMeans regimes + sensor normalization)
- XGBoost: 550 estimators, depth 5, learning rate 0.03, subsample/column sample 0.9, minimum child weight 5, L2 regularization 7

**Safety rails:** uses only the four labelled training trajectories (`train_FD001`–`train_FD004`); does not read official `RUL_FD00x.txt` labels; saves a timestamped copy of the existing model; validates the new artifact before replacing the active file; writes a JSON record of the final configuration. The final fitted model is for this portfolio benchmark, not a certified aviation maintenance system.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from packaging.version import Version
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

if Version(xgb.__version__) < Version("2.0.0"):
    raise RuntimeError(f"xgboost>=2.0 is required for reg:quantileerror; found {xgb.__version__}.")

RANDOM_STATE = 20261026
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
WINDOW = 10
TARGET_CAP = 250
N_REGIMES = 6
N_CUTOFFS_PER_ENGINE = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_file(filename):
    for path in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {filename}; checked data/raw/, CMAPSSData/, and project root.")


def load_cmapss_file(filename):
    frame = pd.read_csv(locate_file(filename), sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected schema in {filename}: {frame.shape}")
    return frame


def load_training_data():
    parts = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"train_{subset}.txt")
        part["subset"] = subset
        part["engine_key"] = subset + "_" + part["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


train_raw = load_training_data()
print(f"XGBoost version: {xgb.__version__}")
print(f"Training rows: {len(train_raw):,} | training engines: {train_raw['engine_key'].nunique():,}")
print("Only train_FD001–FD004.txt are loaded. Official test trajectories and RUL label files are not read.")

XGBoost version: 3.2.0
Training rows: 160,359 | training engines: 709
Only train_FD001–FD004.txt are loaded. Official test trajectories and RUL label files are not read.


## 1. Build causal rolling features

Rolling means/stds and one-cycle changes are calculated within each engine in chronological order. `RUL_true` is generated only as a supervised target and is excluded from model inputs.

In [2]:
def engineer_base_features(frame, window=WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    final_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = final_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features


ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
base = engineer_base_features(train_raw, WINDOW)
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
assert base[BASE_FEATURE_COLUMNS].notna().all().all()
assert not ({"engine_key", "unit_nr", "RUL_true", "subset"} & set(BASE_FEATURE_COLUMNS))


print(f"Causal base features prepared: {len(base):,} rows | {len(BASE_FEATURE_COLUMNS)} base features")
print("RUL_true is retained only as a training target and is excluded from model inputs.")


Causal base features prepared: 160,359 rows | 92 base features
RUL_true is retained only as a training target and is excluded from model inputs.


## 2. Define the regime-aware feature pipeline

The final regime transformer is fitted on all labelled training engines only after model-selection experiments are complete. The same transformer is stored with the model so the dashboard can apply the identical feature transformation to incoming trajectories.

In [3]:
def make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        stats[col] = (float(np.mean(vals)), max(float(np.std(vals, ddof=0)), 1e-6)) if len(vals) else (0.0, 1.0)
    return stats


def fit_regime_transformer(features, fit_engine_keys, random_state=RANDOM_STATE):
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No data supplied to fit operating-regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    kmeans = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    fit["_regime"] = kmeans.fit_predict(scaler.transform(fit[SETTING_COLS])).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = make_stats(group, source_columns)
    subset_stats = {str(s): make_stats(g, source_columns) for s, g in fit.groupby("subset", sort=True)}
    global_stats = make_stats(fit, source_columns)
    return {"setting_scaler": scaler, "kmeans": kmeans, "group_stats": group_stats,
            "subset_stats": subset_stats, "global_stats": global_stats, "n_regimes": N_REGIMES,
            "source_columns": source_columns, "rolling_window": WINDOW,
            "method": "KMeans operating regimes plus subset/regime sensor z-scores"}


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    regime = transformer["kmeans"].predict(transformer["setting_scaler"].transform(out[SETTING_COLS])).astype(int)
    out["_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means, stds = np.empty(len(out), dtype=float), np.empty(len(out), dtype=float)
        for key, locs in positions.items():
            stats = transformer["group_stats"].get(key, transformer["subset_stats"].get(key[0], transformer["global_stats"]))
            mean, sd = stats.get(source_col, transformer["global_stats"][source_col])
            means[locs], stds[locs] = mean, max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_regime"], inplace=True)
    return out


def feature_view(fit_engine_keys):
    transformer = fit_regime_transformer(base, fit_engine_keys)
    augmented = add_regime_features(base, transformer)
    regime_columns = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]]
    columns = BASE_FEATURE_COLUMNS + regime_columns
    if augmented[columns].isna().any().any():
        raise ValueError("Non-finite features after regime transform.")
    return augmented, columns, transformer

print("Regime-aware feature functions are ready. Final transformer will be fitted on all labelled training engines.")


Regime-aware feature functions are ready. Final transformer will be fitted on all labelled training engines.


## 3. Freeze training objective and model parameters

These helper definitions match the loss-function and target-cap experiments. No official test set is used in this stage.

In [4]:
def capped_target(y, cap=TARGET_CAP):
    y = np.asarray(y, dtype=float)
    return y if cap is None else np.minimum(y, float(cap))


def nasa_penalty_array(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    exponent = np.where(error < 0, -error / 13.0, error / 10.0)
    return np.expm1(np.clip(exponent, 0.0, 50.0))


def score_points(points, predictions):
    scored = points[["subset", "engine_key", "unit_nr", "cutoff_cycle", "true_rul_cycles"]].copy().reset_index(drop=True)
    scored["predicted_rul_cycles"] = np.maximum(0.0, np.asarray(predictions, dtype=float))
    scored["error_pred_minus_true"] = scored["predicted_rul_cycles"] - scored["true_rul_cycles"]
    scored["absolute_error"] = scored["error_pred_minus_true"].abs()
    scored["squared_error"] = scored["error_pred_minus_true"] ** 2
    scored["nasa_penalty"] = nasa_penalty_array(scored["true_rul_cycles"], scored["predicted_rul_cycles"])
    scored["within_10_cycles"] = scored["absolute_error"] <= 10
    scored["within_20_cycles"] = scored["absolute_error"] <= 20
    scored["within_30_cycles"] = scored["absolute_error"] <= 30

    # First aggregate repeated cutoffs within each engine, then average engines equally within each subset.
    engine_metrics = scored.groupby(["subset", "engine_key"], as_index=False).agg(
        cases=("engine_key", "size"), mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        nasa_score_per_case=("nasa_penalty", "mean"),
        late_pct=("error_pred_minus_true", lambda x: float(np.mean(x > 0) * 100)),
        mean_bias=("error_pred_minus_true", "mean"),
        within_10_pct=("within_10_cycles", lambda x: float(np.mean(x) * 100)),
        within_20_pct=("within_20_cycles", lambda x: float(np.mean(x) * 100)),
        within_30_pct=("within_30_cycles", lambda x: float(np.mean(x) * 100)),
    )
    by_subset = engine_metrics.groupby("subset", as_index=False).agg(
        engines=("engine_key", "nunique"), cases=("cases", "sum"), mae=("mae", "mean"),
        rmse=("rmse", "mean"), nasa_score_per_case=("nasa_score_per_case", "mean"),
        late_pct=("late_pct", "mean"), mean_bias=("mean_bias", "mean"),
        within_10_pct=("within_10_pct", "mean"), within_20_pct=("within_20_pct", "mean"),
        within_30_pct=("within_30_pct", "mean"),
    )
    summary = {
        "n_cases": int(len(scored)), "n_engines": int(scored["engine_key"].nunique()),
        "pooled_mae": float(scored["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(scored["squared_error"].mean())),
        "pooled_r2": float(r2_score(scored["true_rul_cycles"], scored["predicted_rul_cycles"])),
        "macro_subset_mae": float(by_subset["mae"].mean()),
        "macro_subset_rmse": float(by_subset["rmse"].mean()),
        "macro_subset_nasa_score_per_case": float(by_subset["nasa_score_per_case"].mean()),
        "macro_subset_late_pct": float(by_subset["late_pct"].mean()),
        "macro_subset_mean_bias": float(by_subset["mean_bias"].mean()),
        "macro_subset_within_10_pct": float(by_subset["within_10_pct"].mean()),
        "macro_subset_within_20_pct": float(by_subset["within_20_pct"].mean()),
        "macro_subset_within_30_pct": float(by_subset["within_30_pct"].mean()),
    }
    return summary, by_subset, scored, engine_metrics


def sample_weights(training_frame, policy):
    if policy == "row_uniform":
        return None
    if policy != "engine_balanced":
        raise ValueError(f"Unknown sample-weight policy: {policy}")
    per_engine_rows = training_frame.groupby("engine_key")["engine_key"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / np.maximum(per_engine_rows, 1.0)
    return weights / weights.mean()

FIXED_XGB_PARAMS = {
    "n_estimators": 550,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 5,
    "reg_lambda": 7.0,
    "tree_method": "hist",
    "n_jobs": 4,
}

LOSS_CANDIDATES = [
    {"loss_id": "squarederror", "objective": "reg:squarederror", "objective_params": {}},
    {"loss_id": "pseudohuber_slope10", "objective": "reg:pseudohubererror", "objective_params": {"huber_slope": 10.0}},
    {"loss_id": "absoluteerror", "objective": "reg:absoluteerror", "objective_params": {}},
    {"loss_id": "quantile_q035", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.35}},
    {"loss_id": "quantile_q050", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.50}},
]
WEIGHT_POLICIES = ["row_uniform", "engine_balanced"]

def weighted_quantile(values, quantile, weights=None):
    values = np.asarray(values, dtype=float)
    if weights is None:
        return float(np.quantile(values, quantile))
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]
    cumulative = np.cumsum(sorted_weights)
    cutoff = float(quantile) * float(cumulative[-1])
    pos = min(int(np.searchsorted(cumulative, cutoff, side="left")), len(sorted_values) - 1)
    return float(sorted_values[pos])


def initial_base_score(candidate, y_fit, weights):
    """Set a loss-appropriate starting level; needed for stable robust-loss initialization."""
    objective = candidate["objective"]
    if objective == "reg:squarederror":
        return float(np.average(y_fit, weights=weights)) if weights is not None else float(np.mean(y_fit))
    if objective in {"reg:pseudohubererror", "reg:absoluteerror"}:
        return weighted_quantile(y_fit, 0.50, weights)
    if objective == "reg:quantileerror":
        q = float(candidate["objective_params"].get("quantile_alpha", 0.50))
        return weighted_quantile(y_fit, q, weights)
    raise ValueError(f"Unsupported objective in this experiment: {objective}")


def make_model(candidate, y_fit, weights=None, seed=RANDOM_STATE):
    params = dict(FIXED_XGB_PARAMS)
    params.update(candidate["objective_params"])
    params["objective"] = candidate["objective"]
    params["random_state"] = int(seed)
    # Explicit loss-appropriate intercept avoids unstable automatic initialization
    # for robust losses on RUL values measured in hundreds of cycles.
    params["base_score"] = initial_base_score(candidate, y_fit, weights)
    return XGBRegressor(**params)


def fit_candidate(candidate, fit_frame, feature_columns, fit_engine_keys, seed=RANDOM_STATE):
    mask = fit_frame["engine_key"].isin(set(fit_engine_keys))
    fit_rows = fit_frame.loc[mask]
    X_fit = fit_rows[feature_columns]
    y_fit = capped_target(fit_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
    w = sample_weights(fit_rows, candidate["weight_policy"])
    model = make_model(candidate, y_fit, weights=w, seed=seed)
    if w is None:
        model.fit(X_fit, y_fit)
    else:
        model.fit(X_fit, y_fit, sample_weight=w)
    return model


## 4. Train on all labelled training trajectories

After choosing the model configuration using engine-level validation, refit the selected learner on all available labelled training trajectories. This is the final fit stage, not a new evaluation split.

In [5]:
# Freeze the selected deployment configuration. No candidate search happens in this notebook.
TARGET_CAP = 250
FINAL_CANDIDATE = {
    "loss_id": "pseudohuber_slope10",
    "objective": "reg:pseudohubererror",
    "objective_params": {"huber_slope": 10.0},
    "weight_policy": "row_uniform",
}
RANDOM_STATE = 20261026

ALL_ENGINE_KEYS = sorted(base["engine_key"].unique().tolist())
FINAL_FEATURES, FINAL_FEATURE_COLUMNS, FINAL_TRANSFORMER = feature_view(ALL_ENGINE_KEYS)
final_rows = FINAL_FEATURES.loc[FINAL_FEATURES["engine_key"].isin(ALL_ENGINE_KEYS)].copy()
X_all = final_rows[FINAL_FEATURE_COLUMNS]
y_all = capped_target(final_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
final_weights = sample_weights(final_rows, FINAL_CANDIDATE["weight_policy"])

print(f"Final training engines: {len(ALL_ENGINE_KEYS):,}")
print(f"Final training rows: {len(final_rows):,}")
print(f"Feature count: {len(FINAL_FEATURE_COLUMNS)}")
print(f"Target cap: {TARGET_CAP} cycles")
print(f"Objective: {FINAL_CANDIDATE['objective']} | Huber slope: 10 | weighting: row_uniform")
print("The official test RUL label files are not loaded or used for selection.")

final_model = make_model(FINAL_CANDIDATE, y_all, weights=final_weights, seed=RANDOM_STATE)
if final_weights is None:
    final_model.fit(X_all, y_all)
else:
    final_model.fit(X_all, y_all, sample_weight=final_weights)

print("Final model fit complete. Running artifact preflight before replacing the active model...")


Final training engines: 709
Final training rows: 160,359
Feature count: 140
Target cap: 250 cycles
Objective: reg:pseudohubererror | Huber slope: 10 | weighting: row_uniform
The official test RUL label files are not loaded or used for selection.
Final model fit complete. Running artifact preflight before replacing the active model...


## 5. Validate and save the deployment artifact

The new artifact is written to a temporary file and reloaded for a finite-output smoke test. Only after that check passes is it moved into `models/xgb_rul_model_all_subsets.joblib`. A copy of the previous artifact is retained when one exists.

In [6]:
from datetime import datetime
import os

MODEL_PATH = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
REPORT_PATH = REPORTS_DIR / "final_model_training_summary.json"
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = None
if MODEL_PATH.exists():
    backup_path = MODELS_DIR / f"xgb_rul_model_pre_cap250_{stamp}.joblib"

artifact = {
    "model": final_model,
    "sensor_cols": SENSOR_COLS,
    "setting_cols": SETTING_COLS,
    "feature_columns": FINAL_FEATURE_COLUMNS,
    "base_feature_columns": BASE_FEATURE_COLUMNS,
    "rolling_window": int(WINDOW),
    "rul_cap": int(TARGET_CAP),
    "best_params": {**FIXED_XGB_PARAMS, "objective": FINAL_CANDIDATE["objective"], **FINAL_CANDIDATE["objective_params"]},
    "dataset": "NASA PCoE C-MAPSS FD001-FD004 pooled XGBoost RUL model",
    "subsets": SUBSETS,
    "target_unit": "operating cycles",
    "random_state": RANDOM_STATE,
    "n_training_rows": int(len(train_raw)),
    "n_training_engines": int(train_raw["engine_key"].nunique()),
    "n_model_features": int(len(FINAL_FEATURE_COLUMNS)),
    "feature_mode": "regime_aware",
    "regime_transformer": FINAL_TRANSFORMER,
    "selected_objective": FINAL_CANDIDATE["objective"],
    "objective_id": FINAL_CANDIDATE["loss_id"],
    "objective_specific_params": FINAL_CANDIDATE["objective_params"],
    "sample_weight_policy": FINAL_CANDIDATE["weight_policy"],
    "sample_weight_policy_description": "row_uniform = equal row weights; no sample_weight vector is passed.",
    "experiment_selection": {
        "candidate_id": "cap_250__pseudohuber_slope10__row_uniform",
        "selection_metric": "mean macro-subset MAE across three engine-level validation splits; macro-subset RMSE as corroborating metric; NASA-style asymmetric score reported as a trade-off",
        "validation_splits": [20261026, 20261027, 20261028],
        "validation_macro_subset_mae_mean": 27.083,
        "validation_macro_subset_mae_std": 3.524,
        "validation_macro_subset_rmse_mean": 31.385,
        "validation_macro_subset_rmse_std": 4.034,
        "validation_macro_subset_nasa_score_per_case_mean": 8535621.094,
        "note": "Cap 250 had the lowest macro-subset MAE on each of the three repeated validation splits. Cap 350 had the lowest mean NASA-style score, so the selected configuration reflects an explicit MAE-first decision, not a universal win on every metric.",
        "official_test_rul_labels_used_for_selection": False,
        "official_test_note": "Official test labels/results were inspected earlier in project development; any official-test numbers are exploratory benchmark results, not a pristine blind test.",
    },
}

# Save to a temporary artifact and validate it before replacing the active model.
temp_path = MODELS_DIR / "xgb_rul_model_all_subsets.joblib.tmp"
joblib.dump(artifact, temp_path)
preflight = joblib.load(temp_path)
probe = preflight["model"].predict(X_all.iloc[:20][preflight["feature_columns"]])
if len(probe) != min(20, len(X_all)) or not np.isfinite(probe).all():
    temp_path.unlink(missing_ok=True)
    raise RuntimeError("Artifact preflight failed: prediction output length or finite-value check failed.")
required_keys = {"model", "sensor_cols", "setting_cols", "feature_columns", "rolling_window", "feature_mode", "regime_transformer", "rul_cap"}
if not required_keys.issubset(preflight):
    temp_path.unlink(missing_ok=True)
    raise RuntimeError(f"Artifact preflight failed: missing keys {sorted(required_keys - set(preflight))}")

if backup_path is not None:
    shutil.copy2(MODEL_PATH, backup_path)
os.replace(temp_path, MODEL_PATH)

summary = {
    "model_artifact": str(MODEL_PATH),
    "previous_model_snapshot": str(backup_path) if backup_path else None,
    "dataset": artifact["dataset"],
    "training_rows": artifact["n_training_rows"],
    "training_engines": artifact["n_training_engines"],
    "feature_count": artifact["n_model_features"],
    "target_cap_cycles": TARGET_CAP,
    "objective": FINAL_CANDIDATE["objective"],
    "objective_params": FINAL_CANDIDATE["objective_params"],
    "sample_weight_policy": FINAL_CANDIDATE["weight_policy"],
    "rolling_window_cycles": WINDOW,
    "feature_mode": "regime_aware",
    "saved_artifact_preflight": "passed",
    "official_test_rul_labels_loaded": False,
    "selection_rationale": artifact["experiment_selection"],
}
REPORT_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("\nFINAL MODEL SAVED:", MODEL_PATH)
print("Backup of previous model:", backup_path)
print("Target cap:", TARGET_CAP)
print("Artifact preflight: PASSED")
print("Training summary:", REPORT_PATH)
print("Next: run `streamlit run app.py` and verify the dashboard loads this artifact.")



FINAL MODEL SAVED: d:\industrial_rul_xgboost_project\models\xgb_rul_model_all_subsets.joblib
Backup of previous model: d:\industrial_rul_xgboost_project\models\xgb_rul_model_pre_cap250_20261026_124732.joblib
Target cap: 250
Artifact preflight: PASSED
Training summary: d:\industrial_rul_xgboost_project\reports\final_model_training_summary.json
Next: run `streamlit run app.py` and verify the dashboard loads this artifact.


## 6. Next step

Once the notebook completes successfully, launch the dashboard from the project root:

```powershell
streamlit run app.py
```

Confirm that the dashboard loads the new model and that the Model & Methodology panel reflects the expected regime-aware setup. Do not use the official test benchmark to continue tuning; test labels/results were already inspected earlier in the project, so any subsequent figures should be described as exploratory benchmark reporting.